In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from google.colab import drive
from sklearn.decomposition import PCA

In [64]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [65]:
%cd /content/drive/MyDrive/RIESGO

/content/drive/MyDrive/RIESGO


In [66]:
ls

label_encoders.joblib  minmax_scaler.joblib  riesgo.xlsx


In [67]:
df = pd.read_excel('riesgo.xlsx')
df

,Customer_ID,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,CUS_0x1000,Alistair Barrf,17.375000,913-74-1218,Lawyer,30625.940,2706.161667,6.0,5.0,27,...,Bad,1562.91,33.477546,10.458333,Yes,42.941090,158.549735,High_spent_Medium_value_payments,335.375341,0
1,CUS_0x1009,Arunah,25.750000,063-67-6938,Mechanic,52312.680,4250.390000,6.0,5.0,17,...,Standard,202.68,29.839984,30.714286,Yes,108.366467,146.679378,High_spent_Medium_value_payments,428.743155,1
2,CUS_0x100b,Shirboni,18.500000,238-62-0395,Media_Manager,113781.390,9549.782500,1.0,4.0,1,...,Good,1030.20,34.841449,15.571429,No,0.000000,505.386526,High_spent_Large_value_payments,781.229776,0
3,CUS_0x1011,Schneyerh,43.875000,793-05-8223,Doctor,58918.470,5208.872500,3.0,3.0,17,...,Standard,473.14,27.655897,15.541667,Yes,123.434939,311.060914,Low_spent_Medium_value_payments,332.642837,1
4,CUS_0x1013,Cameront,43.750000,930-49-9615,Mechanic,98620.980,7962.415000,3.0,3.0,6,...,Good,1233.51,31.933940,17.535714,No,228.018084,355.442408,High_spent_Medium_value_payments,472.781009,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12495,CUS_0xff3,Somervilled,55.000000,726-35-5322,Scientist,17032.785,1176.398750,0.0,6.0,2,...,Good,1229.08,32.889398,17.000000,No,33.299764,83.918549,Low_spent_Small_value_payments,271.671562,1
12496,CUS_0xff4,Poornimaf,36.857143,655-05-7666,Entrepreneur,25546.260,2415.855000,8.0,7.0,14,...,Standard,758.44,32.598257,18.440476,Yes,101.328637,152.775690,Low_spent_Small_value_payments,259.981173,1
12497,CUS_0xff6,Shieldsb,18.625000,541-92-8371,Doctor,117639.920,9727.326667,5.0,6.0,1,...,Good,338.30,33.258053,24.625000,No,126.638453,655.228203,High_spent_Small_value_payments,667.322417,1
12498,CUS_0xffc,Brads,17.375000,226-86-7294,Musician,60877.170,5218.097500,6.0,8.0,27,...,Bad,1300.13,34.722108,12.861111,Yes,272.809169,156.172974,High_spent_Large_value_payments,339.951771,0


In [68]:
#Verificar si hay valores nulos
print("\nValores nulos por columna:")
print(df.isnull().sum())


Valores nulos por columna:
Customer_ID                    0
Name                           0
Age                            0
SSN                            0
Occupation                     0
Annual_Income                  0
Monthly_Inhand_Salary          0
Num_Bank_Accounts              0
Num_Credit_Card                0
Interest_Rate                  0
Num_of_Loan                    0
Type_of_Loan                1426
Delay_from_due_date            0
Num_of_Delayed_Payment         0
Changed_Credit_Limit           0
Num_Credit_Inquiries           0
Credit_Mix                     0
Outstanding_Debt               0
Credit_Utilization_Ratio       0
Credit_History_Age             0
Payment_of_Min_Amount          0
Total_EMI_per_month            0
Amount_invested_monthly        0
Payment_Behaviour              0
Monthly_Balance                0
Credit_Score                   0
dtype: int64


In [69]:
#Estadiscias descriptivas de las columnas numericas
print("\nEstadísticas descriptivas: ")
df.describe().T


Estadísticas descriptivas: 


,count,mean,std,min,25%,50%,75%,max
Age,12500.0,33.311294,10.760177,14.000000,24.415179,33.000000,41.750000,56.000000
Annual_Income,12500.0,50505.123449,38300.762656,7005.930000,19342.972500,36999.705000,71683.470000,179987.280000
Monthly_Inhand_Salary,12500.0,4198.468568,3187.142979,303.645417,1625.744479,3097.016667,5961.664375,15204.633333
Num_Bank_Accounts,12500.0,5.368828,2.592493,0.000000,3.000000,5.375000,7.000000,10.500000
Num_Credit_Card,12500.0,5.533620,2.066040,0.500000,4.000000,5.000000,7.000000,10.875000
Interest_Rate,12500.0,14.532080,8.741636,1.000000,7.000000,13.000000,20.000000,34.000000
Num_of_Loan,12500.0,3.532880,2.446442,0.000000,2.000000,3.000000,5.000000,9.000000
Delay_from_due_date,12500.0,21.068780,14.772965,-2.000000,9.875000,17.875000,28.000000,63.250000
Num_of_Delayed_Payment,12500.0,13.338642,6.153148,0.000000,9.000000,13.750000,18.175000,26.375000
Changed_Credit_Limit,12500.0,10.465068,6.445141,0.500000,5.493750,9.370000,14.656250,31.115000


In [70]:
#Ver el tipo de datos de cada columna
print("\nTipos de datos por columna:")
print(df.columns)


Tipos de datos por columna:
Index(['Customer_ID', 'Name', 'Age', 'SSN', 'Occupation', 'Annual_Income',
       'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card',
       'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan', 'Delay_from_due_date',
       'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt',
       'Credit_Utilization_Ratio', 'Credit_History_Age',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score'],
      dtype='object')


In [71]:
df.drop(columns=['Customer_ID', 'Name', 'SSN', 'Type_of_Loan'], axis=1, inplace=True)

In [72]:
#Eliminar las filas donde el valor de 'Credit_Score' sea nulo
df.dropna(subset=['Credit_Score'], inplace=True)

#Restablecer el indice del DataFrame, eliminando cualquier hueco dejado por las filas eliminadas
df.reset_index(drop=True, inplace=True)

In [73]:
#Filtrar columnas de tipo 'object' 
objetc_columns = df.select_dtypes(include=['object']).columns

#Mostrar la frecuencia de valores para cada columna de tipo 'object'
for column in objetc_columns:
    print(f"\nFrecuencia de valores para la columna '{column}':")
    print(df[column].value_counts())
    print("\n")


Frecuencia de valores para la columna 'Occupation':
Occupation
Lawyer           887
Engineer         858
Architect        853
Mechanic         847
Scientist        843
Accountant       843
Media_Manager    840
Developer        840
Teacher          834
Entrepreneur     831
Doctor           821
Journalist       817
Manager          804
Musician         794
Writer           788
Name: count, dtype: int64



Frecuencia de valores para la columna 'Credit_Mix':
Credit_Mix
Standard    5731
Good        3798
Bad         2971
Name: count, dtype: int64



Frecuencia de valores para la columna 'Payment_of_Min_Amount':
Payment_of_Min_Amount
Yes    7360
No     5031
NM      109
Name: count, dtype: int64



Frecuencia de valores para la columna 'Payment_Behaviour':
Payment_Behaviour
Low_spent_Small_value_payments      3860
High_spent_Medium_value_payments    3086
High_spent_Large_value_payments     2726
Low_spent_Medium_value_payments     1136
High_spent_Small_value_payments      972
Low_spent_Large_v

In [74]:
# Contar valores en cada columna del dataframe
print("Conteo de valores por columna:")
print("="*50)

for column in df.columns:
    print(f"\n'{column}':")
    print(f"  Total de valores: {df[column].count()}")
    print(f"  Valores únicos: {df[column].nunique()}")
    print(f"  Valores nulos: {df[column].isnull().sum()}")

Conteo de valores por columna:

'Age':
  Total de valores: 12500
  Valores únicos: 659
  Valores nulos: 0

'Occupation':
  Total de valores: 12500
  Valores únicos: 15
  Valores nulos: 0

'Annual_Income':
  Total de valores: 12500
  Valores únicos: 12488
  Valores nulos: 0

'Monthly_Inhand_Salary':
  Total de valores: 12500
  Valores únicos: 12491
  Valores nulos: 0

'Num_Bank_Accounts':
  Total de valores: 12500
  Valores únicos: 107
  Valores nulos: 0

'Num_Credit_Card':
  Total de valores: 12500
  Valores únicos: 103
  Valores nulos: 0

'Interest_Rate':
  Total de valores: 12500
  Valores únicos: 34
  Valores nulos: 0

'Num_of_Loan':
  Total de valores: 12500
  Valores únicos: 10
  Valores nulos: 0

'Delay_from_due_date':
  Total de valores: 12500
  Valores únicos: 518
  Valores nulos: 0

'Num_of_Delayed_Payment':
  Total de valores: 12500
  Valores únicos: 546
  Valores nulos: 0

'Changed_Credit_Limit':
  Total de valores: 12500
  Valores únicos: 5836
  Valores nulos: 0

'Num_Credi

In [75]:
# Análisis de la cardinalidad de la columna 'Credit_Mix'
print("ANÁLISIS DE LA COLUMNA 'Credit_Mix'")
print("="*50)

# Cardinalidad (número de valores únicos)
cardinalidad = df['Credit_Mix'].nunique()
print(f"\nCardinalidad de 'Credit_Mix': {cardinalidad}")
print(f"(Número de valores únicos: {cardinalidad})")

# Mostrar los valores únicos y su frecuencia
print(f"\nValores únicos en 'Credit_Mix':")
print(df['Credit_Mix'].value_counts())

# Información adicional
print(f"\nTotal de registros en 'Credit_Mix': {df['Credit_Mix'].count()}")
print(f"Valores nulos: {df['Credit_Mix'].isnull().sum()}")

ANÁLISIS DE LA COLUMNA 'Credit_Mix'

Cardinalidad de 'Credit_Mix': 3
(Número de valores únicos: 3)

Valores únicos en 'Credit_Mix':
Credit_Mix
Standard    5731
Good        3798
Bad         2971
Name: count, dtype: int64

Total de registros en 'Credit_Mix': 12500
Valores nulos: 0


In [76]:
#Borrar ocupacion y comportamiento de pago
df.drop(columns=['Occupation', 'Payment_Behaviour'], axis=1, inplace=True)

In [77]:
# Mostrar los 10 primeros datos de la columna credit_score
print("10 PRIMEROS DATOS DE LA COLUMNA 'Credit_Score':")
print("="*50)
print(df['Credit_Score'].head(10))

10 PRIMEROS DATOS DE LA COLUMNA 'Credit_Score':
0    0
1    1
2    0
3    1
4    1
5    2
6    0
7    1
8    2
9    1
Name: Credit_Score, dtype: int64


In [78]:

df['Credit_Score'].unique()

array([0, 1, 2])

In [79]:
# Asignar a x los features, borrando la columna 'Credit_Score'
X = df.drop('Credit_Score', axis=1)

In [80]:
# Mostrar información sobre x
print("Features (x):")
print(f"Forma: {X.shape}")
print(f"\nColumnas en x:")
print(X.columns.tolist())
print(f"\nPrimeras filas de x:")
print(X.head())

Features (x):
Forma: (12500, 19)

Columnas en x:
['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']

Primeras filas de x:
      Age  Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  \
0  17.375       30625.94            2706.161667                6.0   
1  25.750       52312.68            4250.390000                6.0   
2  18.500      113781.39            9549.782500                1.0   
3  43.875       58918.47            5208.872500                3.0   
4  43.750       98620.98            7962.415000                3.0   

   Num_Credit_Card  Interest_Rate  Num_of_Loan  Delay_from_due_date  \
0              5.0             27       

In [81]:
X.columns

Index(['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
       'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
       'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt',
       'Credit_Utilization_Ratio', 'Credit_History_Age',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Monthly_Balance'],
      dtype='object')

In [82]:
# Aplicar to_categorical a la columna 'Credit_Score' con 3 clases
Y = to_categorical(df['Credit_Score'], num_classes=3)

# Mostrar información sobre y
print("Etiqueta (y) después de to_categorical:")
print(f"Forma: {Y.shape}")
print(f"\nPrimeras 5 filas de y (one-hot encoding):")
print(Y[:5])
print(f"\nTipo de dato: {type(Y)}")
print(f"Dtype: {Y.dtype}")

Etiqueta (y) después de to_categorical:
Forma: (12500, 3)

Primeras 5 filas de y (one-hot encoding):
[[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 0.]]

Tipo de dato: <class 'numpy.ndarray'>
Dtype: float64


In [83]:
X = df.drop(columns=['Credit_Score'])
Y = df['Credit_Score']

In [84]:
# Identificar columnas categóricas
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Columnas categóricas encontradas: {categorical_cols}")

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Convertir a string por si hay valores faltantes o mixtos
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le
    print(f"Columna '{col}' codificada.\n  Clases: {le.classes_}")

# Guardar el diccionario de codificadores con joblib para persistencia
joblib.dump(encoders, 'label_encoders.joblib')
print("Encoders guardados en 'label_encoders.joblib'")

# Mostrar las primeras filas de X para verificar
display(X.head())

Columnas categóricas encontradas: ['Credit_Mix', 'Payment_of_Min_Amount']
Columna 'Credit_Mix' codificada.
  Clases: ['Bad' 'Good' 'Standard']
Columna 'Payment_of_Min_Amount' codificada.
  Clases: ['NM' 'No' 'Yes']
Encoders guardados en 'label_encoders.joblib'


,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance
0,17.375,30625.94,2706.161667,6.0,5.0,27,2,62.25,25.000000,1.880000,10.875000,0,1562.91,33.477546,10.458333,2,42.941090,158.549735,335.375341
1,25.750,52312.68,4250.390000,6.0,5.0,17,4,7.25,17.857143,9.730000,3.000000,2,202.68,29.839984,30.714286,2,108.366467,146.679378,428.743155
2,18.500,113781.39,9549.782500,1.0,4.0,1,0,13.50,7.375000,10.911429,1.857143,1,1030.20,34.841449,15.571429,1,0.000000,505.386526,781.229776
3,43.875,58918.47,5208.872500,3.0,3.0,17,3,27.25,14.500000,14.170000,7.000000,2,473.14,27.655897,15.541667,2,123.434939,311.060914,332.642837
4,43.750,98620.98,7962.415000,3.0,3.0,6,3,12.50,8.428571,1.705000,3.000000,1,1233.51,31.933940,17.535714,1,228.018084,355.442408,472.781009


In [85]:

scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X)

# Convertir el resultado a DataFrame con las mismas columnas
X = pd.DataFrame(X_scaled, columns=X.columns)

# Guardar el scaler para persistencia
joblib.dump(scaler, 'minmax_scaler.joblib')
print("Scaler guardado en 'minmax_scaler.joblib'")

# Mostrar algunas filas del X escalado
display(X.head())

Scaler guardado en 'minmax_scaler.joblib'


,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance
0,0.080357,0.136547,0.161232,0.571429,0.433735,0.787879,0.222222,0.984674,0.947867,0.045076,0.664122,0.0,0.312671,0.472904,0.305500,1.0,0.028341,0.145282,0.193035
1,0.279762,0.261917,0.264865,0.571429,0.433735,0.484848,0.444444,0.141762,0.677048,0.301486,0.183206,1.0,0.040507,0.257902,0.919206,1.0,0.071522,0.133308,0.267348
2,0.107143,0.617266,0.620505,0.095238,0.337349,0.000000,0.000000,0.237548,0.279621,0.340076,0.113413,0.5,0.206083,0.553520,0.460415,0.5,0.000000,0.495156,0.547895
3,0.711310,0.300105,0.329188,0.285714,0.240964,0.484848,0.333333,0.448276,0.549763,0.446513,0.427481,1.0,0.094623,0.128808,0.459513,1.0,0.081467,0.299129,0.190860
4,0.708333,0.529624,0.513977,0.285714,0.240964,0.151515,0.333333,0.222222,0.319567,0.039360,0.183206,0.5,0.246763,0.381668,0.519928,0.5,0.150491,0.343899,0.302398


In [86]:
# Mostrar los 10 primeros registros del DataFrame original
print("10 primeros registros del DataFrame:")
display(df.head(10))

10 primeros registros del DataFrame:


,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Credit_Score
0,17.375,30625.94,2706.161667,6.0,5.0,27,2,62.250,25.000000,1.880000,10.875000,Bad,1562.91,33.477546,10.458333,Yes,42.941090,158.549735,335.375341,0
1,25.750,52312.68,4250.390000,6.0,5.0,17,4,7.250,17.857143,9.730000,3.000000,Standard,202.68,29.839984,30.714286,Yes,108.366467,146.679378,428.743155,1
2,18.500,113781.39,9549.782500,1.0,4.0,1,0,13.500,7.375000,10.911429,1.857143,Good,1030.20,34.841449,15.571429,No,0.000000,505.386526,781.229776,0
3,43.875,58918.47,5208.872500,3.0,3.0,17,3,27.250,14.500000,14.170000,7.000000,Standard,473.14,27.655897,15.541667,Yes,123.434939,311.060914,332.642837,1
4,43.750,98620.98,7962.415000,3.0,3.0,6,3,12.500,8.428571,1.705000,3.000000,Good,1233.51,31.933940,17.535714,No,228.018084,355.442408,472.781009,1
5,27.000,46951.02,3725.585000,7.0,4.0,16,0,8.000,9.125000,16.401429,7.750000,Standard,340.22,35.182883,21.154762,Yes,0.000000,263.812274,398.217749,2
6,15.000,61194.81,5014.567500,7.0,7.0,23,8,23.375,21.714286,28.630000,8.000000,Bad,2773.09,28.293641,13.958333,Yes,225.368691,225.456923,320.631135,0
7,51.500,170614.28,14463.856667,2.0,6.0,9,2,-1.125,2.250000,0.730000,3.000000,Good,849.69,37.188278,20.380952,No,208.907479,340.627976,1170.605581,1
8,30.375,89064.52,7256.043333,5.0,3.0,1,1,5.375,5.285714,6.620000,3.000000,Good,648.36,33.927329,29.952381,No,37.572751,384.920145,573.111438,2
9,25.750,50807.44,4197.953333,8.0,4.0,11,4,12.125,9.571429,3.100000,4.000000,Standard,869.59,34.275254,22.583333,Yes,88.759919,112.067830,473.967584,1


In [87]:
# Contar registros disponibles para X y Y
print(f"Cantidad de registros en X: {X.shape[0]}")
print(f"Cantidad de registros en Y: {Y.shape[0]}")

Cantidad de registros en X: 12500
Cantidad de registros en Y: 12500


In [88]:
# Dividir X y Y en conjuntos de entrenamiento y prueba (85/15) con stratify y semilla

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.15,
    stratify=Y,
    random_state=42
)

print(f"Tamaño X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Tamaño Y_train: {Y_train.shape}, Y_test: {Y_test.shape}")

Tamaño X_train: (10625, 19), X_test: (1875, 19)
Tamaño Y_train: (10625,), Y_test: (1875,)


In [89]:
ls -la

total 2824
-rw------- 1 root root     763 Mar 12 00:06 label_encoders.joblib
-rw------- 1 root root    2063 Mar 12 00:06 minmax_scaler.joblib
-rw------- 1 root root 2887904 Mar 10 23:43 riesgo.xlsx


In [90]:
# Mostrar las clases (etiquetas) de cada columna codificada con LabelEncoder
for col in encoders:
    print(f"Column '{col}': {encoders[col].classes_}")

Column 'Credit_Mix': ['Bad' 'Good' 'Standard']
Column 'Payment_of_Min_Amount': ['NM' 'No' 'Yes']


In [91]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       12500 non-null  float64
 1   Annual_Income             12500 non-null  float64
 2   Monthly_Inhand_Salary     12500 non-null  float64
 3   Num_Bank_Accounts         12500 non-null  float64
 4   Num_Credit_Card           12500 non-null  float64
 5   Interest_Rate             12500 non-null  float64
 6   Num_of_Loan               12500 non-null  float64
 7   Delay_from_due_date       12500 non-null  float64
 8   Num_of_Delayed_Payment    12500 non-null  float64
 9   Changed_Credit_Limit      12500 non-null  float64
 10  Num_Credit_Inquiries      12500 non-null  float64
 11  Credit_Mix                12500 non-null  float64
 12  Outstanding_Debt          12500 non-null  float64
 13  Credit_Utilization_Ratio  12500 non-null  float64
 14  Credit

In [92]:
# Convert Y_train to 1D for SelectKBest
y_train_1d = np.argmax(Y_train, axis=1)

# Apply SelectKBest to select the top 10 features
selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train, y_train_1d)
X_test_selected = selector.transform(X_test)

# Get the names of the selected features
selected_features = X_train.columns[selector.get_support()].tolist()
print("Selected features:", selected_features)

# Optionally, convert back to DataFrame for consistency
X_train_selected = pd.DataFrame(X_train_selected, columns=selected_features, index=X_train.index)
X_test_selected = pd.DataFrame(X_test_selected, columns=selected_features, index=X_test.index)

ValueError: `axis` must be fewer than the number of dimensions (1)

In [93]:
# Aplicar PCA con 5 componentes a X_train_selected
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_selected)

# Mostrar resultados
print("Varianza explicada por cada componente:")
print(pca.explained_variance_ratio_)
print(f"\nVarianza explicada acumulada: {pca.explained_variance_ratio_.cumsum()}")
print(f"\nForma del conjunto de datos reducido: {X_train_pca.shape}")

NameError: name 'PCA' is not defined

In [94]:
# Guardar el modelo PCA entrenado
joblib.dump(pca, 'pca_model.joblib')
print("Modelo PCA guardado en 'pca_model.joblib'")

# Verificar que se guardó correctamente
pca_loaded = joblib.load('pca_model.joblib')
print(f"Modelo PCA cargado exitosamente")
print(f"Componentes: {pca_loaded.n_components}")
print(f"Varianza explicada acumulada: {pca_loaded.explained_variance_ratio_.cumsum()}")

NameError: name 'pca' is not defined